# 04 - The Three Firewalls: Carrier, Move, and Replay

> **Claim-typing legend**
> - **[Pinned]**: reproduced from a named local script, benchmark, or cryptographic receipt (`code/*.json`).
> - **[Proved]**: a finite check in this notebook. Not a general proof.
> - **[Declared]**: a grammar, carrier alphabet, cost, or objective chosen by the model.
> - **[Open]**: an obligation for which this package supplies no proof or exhaustive decision.

*The 4-Field Packet Kernel, Closed Operators, and Fail-Closed Verification Gates*

---

## 1. The 4-Field Packet Envelope

Communication between the proposer (LLM) and the host verifier occurs via discrete, content-addressed **Packets**:
$$\text{Packet} = (\text{contract}, \text{task}, \text{payload}, \text{parent\_hash})$$

The hash of the packet is computed deterministically via SHA-256 over its canonical JSON serialization:
$$h_t = \text{SHA-256}(\text{CanonicalJson}(\text{Packet}_t))$$

## 2. The Three Sovereign Firewalls

1. **FW-CARRIER**: Asserts that every character in the packet belongs strictly to the declared Unicode codepoint whitelist (ASCII 32..126, Greek alphabet, declared logic arrows $\to, \vdash, \bot$).
2. **FW-MOVE**: Asserts that the payload operation begins with an approved closed verb from `DECLARED_OPERATORS` (ASSERT, CUT, REDUCE, PROVE, VERIFY, RETRACT, REFUSE, LAND).
3. **FW-REPLAY**: Asserts that the parent hash $h_{t-1}$ matches the previous ledger head and that the state transition can be deterministically re-executed in $O(K)$ host cycles.


In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path(".").resolve()))

from notebook_lib import GovernorAdapter

# Test 1: Clean Valid Packet
p1 = GovernorAdapter.validate_packet(
    contract="SYM-032",
    task="PROVE_CYCLE_0",
    payload={"op": "ASSERT", "claim": "x -> y", "evidence": "PIN_01"},
)
print(f"[Pinned] Packet 1 Admitted: {p1.admitted} (Hash: {p1.packet_hash[:16]}...)")

# Test 2: FW-MOVE Violation (Undeclared Verb)
p2 = GovernorAdapter.validate_packet(
    contract="SYM-032", task="PROVE_CYCLE_0", payload={"op": "SPECULATE", "claim": "x -> y"}
)
print(
    f"[Pinned] Packet 2 Admitted: {p2.admitted} | Refusal: {p2.refusal_code} | Reason: {p2.reason}"
)

# Test 3: FW-CARRIER Violation (Forbidden Unicode control character)
p3 = GovernorAdapter.validate_packet(
    contract="SYM-032", task="PROVE_CYCLE_0", payload={"op": "ASSERT", "claim": "x \x00 y"}
)
print(f"[Pinned] Packet 3 Admitted: {p3.admitted} | Refusal: {p3.refusal_code}")

## Anchors

- Former notebook: `04_packet_firewall_host`
- The code checks one finite example. It is not a general proof.

## Exercise

Change one declared input in the code cell. The check should fail for the wrong object and pass for the finite example written here.


In [ ]:
assert p1.admitted is True
assert p2.admitted is False
assert p3.admitted is False